# Diabetes Challenge

Your task today is to **analyze** the Kaggle "Pima Indians Diabetes Database" and to **predict** whether a patient has Diabetes or not.

## Task:
- Load the data from the database. The schema is called `diabetes`. To connect to the database you need to copy the `.env` file from the visualization or hands-on-ml repository into this repo. Explore the database, try to establish what the relationships between the tables are (1-1, 1-N, N-M). Explain to yourself and the group what data do you see and whether it makes sense. What JOINs are appropriate to use and why?
- Use at least two different classification algorithms we have learned so far to predict Diabetes patients.
- Discuss before you start with the modeling process which **evaluation metric** you choose and explain why.
- Implement a GridSearchCV or RandomizedSearchCV to tune the hyperparameters of your model.
- **Optional:** If you have time at the end, try to use sklearn's pipline module to encapsulate all the steps into a pipeline.

Don't forget to split your data in train and test set. And analyze your final model on the test data. It might also be necessary to scale your data in order to improve the performance of some of the models.


## Helpful links and advise:
- [sklearn documentation on hyperparameter tuning](https://scikit-learn.org/stable/modules/grid_search.html#grid-search)
- It might be helpful to check some sources on how to deal with imbalanced data.
    * [8 Tactics to Combat Imbalanced Classes](https://machinelearningmastery.com/tactics-to-combat-imbalanced-classes-in-your-machine-learning-dataset/)
    * [Random-Oversampling/Undersampling](https://machinelearningmastery.com/random-oversampling-and-undersampling-for-imbalanced-classification/)


# Data Description

## Context
This dataset is originally from the National Institute of Diabetes and Digestive and Kidney Diseases. The objective of the dataset is to diagnostically predict whether or not a patient has diabetes, based on certain diagnostic measurements included in the dataset. Several constraints were placed on the selection of these instances from a larger database. In particular, all patients here are females at least 21 years old of Pima Indian heritage.

## Acknowledgements
Smith, J.W., Everhart, J.E., Dickson, W.C., Knowler, W.C., & Johannes, R.S. (1988). Using the ADAP learning algorithm to forecast the onset of diabetes mellitus. In Proceedings of the Symposium on Computer Applications and Medical Care (pp. 261--265). IEEE Computer Society Press.

## About this dataset
The datasets consist of several medical predictor (independent) variables and one target (dependent) variable, Outcome. Independent variables include the number of pregnancies the patient has had, their BMI, insulin level, age, and so on. For the outcome class value 1 is interpreted as "tested positive for diabetes".

|Column Name| Description|
|:------------|:------------|
|Pregnancies|Number of times pregnant|
|Glucose|Plasma glucose concentration a 2 hours in an oral glucose tolerance test|
|BloodPressure|Diastolic blood pressure (mm Hg)|
|SkinThickness|Triceps skin fold thickness (mm)|
|Insulin|2-Hour serum insulin (mu U/ml)|
|BMI|Body mass index (weight in kg/(height in m)^2)|
|DiabetesPedigreeFunction| Diabetes pedigree function|
|Age| Age (years)|
|Outcome|Class variable (0 or 1) |

In [ ]:
print(1+2)

: 

In [ ]:
# Import of relevant packages
import numpy as np
import pandas as pd
import warnings
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import cross_val_predict
from sklearn.metrics import accuracy_score, recall_score, precision_score

from sklearn.linear_model import LogisticRegression

# Set random seed
RSEED = 42
warnings.filterwarnings("ignore")

In [ ]:
# Loading the titanic dataset
df_original = pd.read_csv('data/diabetes.csv')

|Column Name| Description|
|:------------|:------------|
|Pregnancies|Number of times pregnant|
|Glucose|Plasma glucose concentration a 2 hours in an oral glucose tolerance test|
|BloodPressure|Diastolic blood pressure (mm Hg)|
|SkinThickness|Triceps skin fold thickness (mm)|
|Insulin|2-Hour serum insulin (mu U/ml)|
|BMI|Body mass index (weight in kg/(height in m)^2)|
|DiabetesPedigreeFunction| Diabetes pedigree function|
|Age| Age (years)|
|Outcome|Class variable (0 or 1) |

In [ ]:
display(df_original.head(2))
display(df_original.tail(2))

In [ ]:
# Inspecting the type of features
df_original.info()

In [ ]:
# Having a look at some simple, descriptive statistics
df_original.describe().round(2).T

In [ ]:
# How many unique entries do the features have?
df_original.nunique()

In [ ]:
# Define the columns to drop
columns_to_drop = ['id', 'id.1', 'id.2', 'id.3', 'patientid', 'patientid.1', 'patientid.2']

# Drop the specified columns and create a new DataFrame
df_new = df_original.drop(columns=columns_to_drop)

# Display information about the new DataFrame to confirm the changes
display(df_new.head())
df_new.info()

In [ ]:
df_new = df_new.rename(columns={
    'pregnancies': 'Pregnancies',
    'glucose': 'Glucose',
    'bloodpressure': 'BloodPressure',
    'skinthickness': 'SkinThickness',
    'insulin': 'Insulin',
    'bmi': 'BMI',
    'diabetespedigreefunction': 'DiabetesPedigreeFunction',
    'outcome': 'Outcome'
})

# Convert 'measurement_date' to datetime objects
df_new['measurement_date'] = pd.to_datetime(df_new['measurement_date'])

# Extract year, month, and day
df_new['MeasurementYear'] = df_new['measurement_date'].dt.year
df_new['MeasurementMonth'] = df_new['measurement_date'].dt.month
df_new['MeasurementDay'] = df_new['measurement_date'].dt.day

# Drop the original 'measurement_date' column
df_new = df_new.drop(columns=['measurement_date'])

# Define the desired column order
desired_column_order = [
    'Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI',
    'DiabetesPedigreeFunction', 'Age', 'MeasurementYear',
    'MeasurementMonth', 'MeasurementDay', 'Outcome'
]

# Reorder the columns
df_new = df_new[desired_column_order]

# Display the first 5 rows of the transformed DataFrame
display(df_new.head())

# Display the information about the DataFrame
df_new.info()

In [ ]:
# Создаем новый датафрейм data, выкидывая лишние столбцы
data = df_new.drop(['MeasurementYear', 'MeasurementMonth', 'MeasurementDay'], axis=1)
data.head()


# Checking for NaN and duplicates

In [ ]:
data.isna().sum()

In [ ]:
data.duplicated().sum()

# Train-test split

In [ ]:
# Train-test split considering stratification

df_train, df_test = train_test_split(
    data, 
    test_size=0.2,       # 20% — золотой стандарт для малых данных
    random_state=42, 
    stratify=data.Outcome 
)

In [ ]:
# Print shape of datasets
print('Train data')
print('# df_train:     {}'.format(df_train.shape[0]))
print('==================')
print('Test data')
print('# df_test:     {}'.format(df_test.shape[0]))

In [ ]:
fig, axes = plt.subplots(1,2, figsize=(10, 4))
fig.suptitle("Representation of target variable: Train (left) vs. Test (right) data set")
sns.countplot(x=df_train.Outcome, ax=axes[0], palette={'0': 'green', '1': 'red'});
sns.countplot(x=df_test.Outcome, ax=axes[1], palette={'0': 'green', '1': 'red'});

# Outliers

In [ ]:
features = data.drop('Outcome', axis=1).columns

plt.figure(figsize=(15, 20))
for i, col in enumerate(features):
    plt.subplot(4, 2, i + 1) # Создаем подграфик в сетке 4x2
    sns.histplot(data[col], kde=True, color='skyblue') # Рисуем гистограмму с линией распределения
    plt.title(f'Distribution of {col}')
    plt.tight_layout() # Чтобы названия не налезали друг на друга

plt.show()

In [ ]:
import math

# Выцепляем только признаки, Outcome нам в общую кучу не нужен
features = [col for col in data.columns if col != 'Outcome']
n_cols = 3
n_rows = math.ceil(len(features) / n_cols) # Считаем строки с округлением вверх

plt.figure(figsize=(18, n_rows * 5)) # Высоту подгоняем под количество строк

for i, col in enumerate(features):
    ax = plt.subplot(n_rows, n_cols, i + 1)
    
    # Рисуем гистограмму, добавим чуток стиля
    sns.histplot(data[col], kde=True, ax=ax, color='teal', alpha=0.7)
    
    plt.title(f'Distribution: {col}', fontsize=14)
    plt.xlabel('') # Убираем лишние подписи снизу, и так понятно
    plt.ylabel('Count')

plt.tight_layout()
plt.show()

In [ ]:
features = [col for col in data.columns if col != 'Outcome']
n_cols = 3
n_rows = math.ceil(len(features) / n_cols)

plt.figure(figsize=(18, n_rows * 4)) # Высоту чуть уменьшил, чтобы не были слишком "жирными"

for i, col in enumerate(features):
    ax = plt.subplot(n_rows, n_cols, i + 1)
    
    # Секрет успеха: ставим данные в x, и график сам ложится
    sns.boxplot(x=data[col], ax=ax, color='lightsalmon', width=0.6)
    
    plt.title(f'Horizontal Boxplot: {col}', fontsize=14)
    plt.xlabel('Value')
    plt.grid(axis='x', linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()

In [ ]:
# 1. Готовим замес: метим данные
df_tr = df_train.copy()
df_te = df_test.copy()
df_tr['Source'] = 'Train'
df_te['Source'] = 'Test'

# Склеиваем
df_all = pd.concat([df_tr, df_te], axis=0).reset_index(drop=True)

# 2. Настраиваем сетку (без Outcome)
features = [col for col in df_train.columns if col != 'Outcome']
n_cols = 3
n_rows = math.ceil(len(features) / n_cols)

plt.figure(figsize=(18, n_rows * 5))

for i, col in enumerate(features):
    ax = plt.subplot(n_rows, n_cols, i + 1)
    
    # МАГИЯ: x — это значения колонки, y — это наша метка Source
    # Это нарисует два горизонтальных бокса один под другим
    sns.boxplot(data=df_all, x=col, y='Source', ax=ax, palette={'Train': 'skyblue', 'Test': 'salmon'})
    
    plt.title(f'Train vs Test: {col}', fontsize=14)
    plt.xlabel('Value')
    plt.ylabel('') # Убираем лишнюю надпись сбоку
    plt.grid(axis='x', linestyle='--', alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
g = sns.pairplot(data, hue='Outcome', palette='seismic', corner=True, diag_kind='kde')

# Ручной призыв легенды, если она спряталась
g._legend.set_bbox_to_anchor((0.8, 0.7)) # Двигаем её в пустое место справа сверху
g._legend.set_title('Outcome (0=Healthy, 1=Diabetes)') # Подписываем по-человечески

# Делаем шрифт побольше, чтобы не щуриться
plt.setp(g._legend.get_texts(), fontsize='12') 
plt.setp(g._legend.get_title(), fontsize='14')

plt.show()

In [ ]:
# 1. Метим данные, чтобы не перепутать
df_train_tmp = df_train.copy()
df_test_tmp = df_test.copy()
df_train_tmp['Dataset'] = 'Train'
df_test_tmp['Dataset'] = 'Test'

# 2. Склеиваем их в общую кучу
df_combined = pd.concat([df_train_tmp, df_test_tmp])

# 3. Рисуем магию. Теперь hue='Dataset'!
# Выбираем пару-тройку ключевых фич, иначе график будет рисоваться до завтра
features_to_plot = ['Glucose', 'BMI', 'Age', 'Dataset'] 

g = sns.pairplot(
    df_combined[features_to_plot], 
    hue='Dataset', 
    palette={'Train': 'blue', 'Test': 'orange'}, # Разные цвета для наборов
    corner=True, 
    diag_kind='kde',
    plot_kws={'alpha': 0.4, 's': 15} # Делаем точки прозрачными, чтобы видеть наложения
)

# Подтягиваем легенду в пустое место
g._legend.set_bbox_to_anchor((0.8, 0.7))
g._legend.set_title('Dataset Split Check')

plt.suptitle("Comparison of Distribution: Train vs Test", y=1.02, fontsize=16)
plt.show()

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# 1. Готовим данные (копируем и сразу дропаем таргет)
# Делаем это через drop, чтобы в df_all не попал Outcome
df_tr = df_train.drop('Outcome', axis=1).copy()
df_te = df_test.drop('Outcome', axis=1).copy()

# Добавляем метку набора
df_tr['Source'] = 'Train'
df_te['Source'] = 'Test'

# Склеиваем всё в один мега-датафрейм
df_all = pd.concat([df_tr, df_te], axis=0).reset_index(drop=True)

# 2. Настройка визуализации (теперь без лишних колонок)
g = sns.pairplot(
    df_all, 
    hue='Source', 
    palette={'Train': '#1f77b4', 'Test': '#ff7f0e'}, 
    corner=True,             
    diag_kind='kde',         
    plot_kws={'alpha': 0.3, 's': 10, 'edgecolor': 'none'}, 
    diag_kws={'fill': True}  
)

# 3. Тюнинг легенды и заголовка
g._legend.set_bbox_to_anchor((0.7, 0.7))
g._legend.set_title('Data Split Check')
plt.suptitle("Global Train vs Test Distribution Comparison (Features Only)", y=1.02, fontsize=20)

plt.show()

In [ ]:
# Считаем матрицу корреляции
corr_matrix = data.corr()

# Настраиваем размер, чтобы всё влезло
plt.figure(figsize=(12, 10))

# Рисуем хитмап
# annot=True — чтобы цифры были внутри квадратиков
# cmap='coolwarm' — синий (холодно, минус), красный (горячо, плюс)
# fmt=".2f" — округляем до двух знаков после запятой
sns.heatmap(corr_matrix, annot=True, fmt=".2f", cmap='coolwarm', center=0, linewidths=0.5)

plt.title("Correlation Heatmap: Who's the Boss here?", fontsize=16)
plt.show()

In [ ]:
# 1. Готовим данные (без Outcome, сравниваем только фичи)
df_tr = df_train.drop('Outcome', axis=1)
df_te = df_test.drop('Outcome', axis=1)

# Считаем матрицы корреляции
corr_tr = df_tr.corr()
corr_te = df_te.corr()

# 2. Магия маскирования
# Создаем пустую матрицу такого же размера
corr_combined = pd.DataFrame(np.zeros_like(corr_tr), columns=corr_tr.columns, index=corr_tr.index)

# Заполняем нижний треугольник данными из Train (np.tril - triangle lower)
mask_lower = np.tril(np.ones_like(corr_tr, dtype=bool))
corr_combined = corr_combined.where(~mask_lower, corr_tr)

# Заполняем верхний треугольник данными из Test (np.triu - triangle upper)
# Делаем сдвиг k=1, чтобы не перезаписать диагональ
mask_upper = np.triu(np.ones_like(corr_te, dtype=bool), k=1)
corr_combined = corr_combined.where(~mask_upper, corr_te)

# 3. Визуализация
plt.figure(figsize=(14, 12))

# Рисуем хитмап
# fmt=".2f" — два знака после запятой, чтобы не было каши
# linewidths=0.5 — тонкие линии между ячейками для четкости
sns.heatmap(corr_combined, annot=True, fmt=".2f", cmap='coolwarm', 
            center=0, linewidths=0.5, square=True, cbar_kws={"shrink": .8})

# Добавляем поясняющие надписи, чтобы не запутаться
plt.text(0.5, -0.2, 'Нижний треугольник: TRAIN', fontsize=12, color='blue', transform=plt.gca().transAxes, ha='center')
plt.text(0.5, -0.5, 'Верхний треугольник: TEST', fontsize=12, color='red', transform=plt.gca().transAxes, ha='center')

plt.title("Correlation Battle: Train (Lower) vs Test (Upper)", fontsize=18, y=1.03)
plt.show()

# Modelling

# Train test split

# Exploration and Cleaning